# LeetCode 1450 — Number of Students Doing Homework at a Given Time bằng Python

**Link đề:** https://leetcode.com/problems/number-of-students-doing-homework-at-a-given-time/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def busyStudent(self, startTime: list[int], endTime: list[int], queryTime: int) -> int:
        ...
```

### Mục tiêu học
1. Mô hình hóa dữ liệu thành **đoạn đóng** `[start, end]` và kiểm tra "điểm nằm trong đoạn".
2. Dùng thành thạo **so sánh chuỗi** `a <= q <= b`, **`zip`** để duyệt song song hai mảng, `sum` trên giá trị `bool`.
3. Đi qua 5 lời giải: mô phỏng dòng thời gian → duyệt chỉ số → `zip` + `sum` → **mảng hiệu (difference array) + prefix sum** → **sắp xếp + `bisect`**.
4. Hiểu khi nào cần cách "chuẩn bị trước" (tiền xử lý) để trả lời **nhiều truy vấn** nhanh.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (số học sinh bận lớn nhất, *Car Pooling*).

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho hai mảng số nguyên `startTime`, `endTime` và số nguyên `queryTime`. Học sinh thứ `i` bắt đầu làm bài lúc `startTime[i]` và xong lúc `endTime[i]`.

Trả về số học sinh **đang làm bài** tại thời điểm `queryTime`, tức số `i` sao cho `queryTime` nằm trong đoạn `[startTime[i], endTime[i]]` **tính cả hai đầu** (inclusive).

| Ví dụ | `startTime` | `endTime` | `queryTime` | Output |
|---|---|---|---:|---:|
| 1 | `[1,2,3]` | `[3,2,7]` | 4 | 1 |
| 2 | `[4]` | `[4]` | 4 | 1 |

Giải thích ví dụ 1: học sinh 0 làm `[1,3]` (xong trước 4), học sinh 1 làm `[2,2]`, học sinh 2 làm `[3,7]` — chỉ học sinh 2 đang làm lúc 4.

**Constraints**
- `startTime.length == endTime.length`
- `1 <= startTime.length <= 100`
- `1 <= startTime[i] <= endTime[i] <= 1000`
- `1 <= queryTime <= 1000`

Câu hỏi gợi mở:
- Ví dụ 2: bắt đầu và kết thúc **cùng lúc 4**, hỏi lúc 4 → có tính không? (Có, vì inclusive.)
- Hai mảng có cùng độ dài; phần tử cùng chỉ số `i` thuộc **cùng một học sinh**. Làm sao duyệt chúng "theo cặp"?

In [1]:
startTime, endTime, queryTime = [1, 2, 3], [3, 2, 7], 4
for i in range(len(startTime)):
    print(f"học sinh {i}: đoạn [{startTime[i]}, {endTime[i]}]")
print("queryTime =", queryTime)

học sinh 0: đoạn [1, 3]
học sinh 1: đoạn [2, 2]
học sinh 2: đoạn [3, 7]
queryTime = 4


## 2. Nền tảng Python cần dùng

### 2.1. Đoạn đóng và so sánh chuỗi `a <= q <= b`

"`q` nằm trong đoạn `[a, b]`" ⇔ `a <= q` **và** `q <= b`. Python cho phép viết gọn `a <= q <= b` (chained comparison) — tương đương `a <= q and q <= b`.

In [2]:
a, b_ = 3, 7
for q in [2, 3, 5, 7, 8]:
    print(q, "trong [3,7]?", a <= q <= b_, "| viết dài:", a <= q and q <= b_)

2 trong [3,7]? False | viết dài: False
3 trong [3,7]? True | viết dài: True
5 trong [3,7]? True | viết dài: True
7 trong [3,7]? True | viết dài: True
8 trong [3,7]? False | viết dài: False


Lưu ý đầu mút: `3` và `7` **có** thuộc đoạn đóng `[3, 7]`. Nếu viết `a < q < b` (dấu `<`) ta sẽ bỏ sót hai đầu — lỗi phổ biến nhất của bài.

### 2.2. `zip` — duyệt song song hai (hay nhiều) list

`zip(startTime, endTime)` sinh cặp `(startTime[0], endTime[0]), (startTime[1], endTime[1]), ...`. Kết hợp unpacking: `for s, e in zip(startTime, endTime)`.

In [3]:
print(list(zip([1, 2, 3], [3, 2, 7])))
for s, e in zip([1, 2, 3], [3, 2, 7]):
    print("start =", s, "end =", e)

[(1, 3), (2, 2), (3, 7)]
start = 1 end = 3
start = 2 end = 2
start = 3 end = 7


### 2.3. `bool` là số: `True == 1`, `False == 0`

Vì vậy `sum(điều_kiện for ...)` **đếm** số lần điều kiện đúng.

In [4]:
print(True + True + False)
flags = [s <= 4 <= e for s, e in zip([1, 2, 3], [3, 2, 7])]
print(flags, "→ sum =", sum(flags))

2
[False, False, True] → sum = 1


### 2.4. Prefix sum (tổng tiền tố)

Với list `a`, `prefix[t] = a[0] + a[1] + ... + a[t]`. Ta có thể tính trong một lượt: `prefix[t] = prefix[t-1] + a[t]`. Hàm `itertools.accumulate` làm sẵn việc này.

In [5]:
from itertools import accumulate
a = [0, 1, 0, -1, 2]
print(list(accumulate(a)))   # [0, 1, 1, 0, 2]
running = 0
for t, x in enumerate(a):
    running += x
    print(f"t={t}: cộng {x:>2} → prefix = {running}")

[0, 1, 1, 0, 2]
t=0: cộng  0 → prefix = 0
t=1: cộng  1 → prefix = 1
t=2: cộng  0 → prefix = 1
t=3: cộng -1 → prefix = 0
t=4: cộng  2 → prefix = 2


### 2.5. `bisect` — tìm vị trí chèn trong list đã sắp xếp

Với list `xs` **đã sắp xếp tăng**:
- `bisect_right(xs, q)` = số phần tử `<= q`.
- `bisect_left(xs, q)` = số phần tử `< q`.

Cả hai chạy `O(log n)` nhờ tìm kiếm nhị phân.

In [6]:
from bisect import bisect_left, bisect_right
xs = [1, 2, 2, 3, 7]
print("số phần tử <= 2:", bisect_right(xs, 2))   # 3
print("số phần tử <  2:", bisect_left(xs, 2))    # 1
print("số phần tử <= 4:", bisect_right(xs, 4))   # 4

số phần tử <= 2: 3
số phần tử <  2: 1
số phần tử <= 4: 4


## 3. Cách 1 — Mô phỏng dòng thời gian (trực giác "vẽ lịch")

Cách người mới hay nghĩ khi "nhìn" bài toán: vẽ một trục thời gian `0..1000`, với mỗi học sinh tô **mọi** thời điểm từ `start` tới `end` (cộng 1 vào ô đó). Cuối cùng đọc ô `queryTime`.

In [7]:
def busy_timeline(startTime, endTime, queryTime):
    timeline = [0] * 1001                     # thời điểm 0..1000
    for s, e in zip(startTime, endTime):
        for t in range(s, e + 1):             # e + 1 vì range không gồm đầu cuối
            timeline[t] += 1
    return timeline[queryTime]

print(busy_timeline([1, 2, 3], [3, 2, 7], 4))   # 1
print(busy_timeline([4], [4], 4))               # 1

1
1


### 3.1. Vấn đề của cách mô phỏng

- Tốn `O(n · T)` thời gian với `T` = độ dài đoạn (tới 1000) và `O(T)` bộ nhớ.
- Ta tính số học sinh bận ở **mọi** thời điểm, trong khi đề chỉ hỏi **một** thời điểm.

Đếm số lần cộng với 100 học sinh đều làm `[1, 1000]`:

In [8]:
n = 100
ops = sum(e - s + 1 for s, e in zip([1] * n, [1000] * n))
print("Số phép cộng vào timeline:", ops)

Số phép cộng vào timeline: 100000


## 4. Nút thắt và insight

**Nút thắt:** tô cả đoạn dài cho mỗi học sinh, trong khi chỉ có **một điểm** quan trọng.

**Insight:** với một học sinh, câu hỏi "có bận lúc `q` không?" trả lời bằng **một phép so sánh** `start <= q <= end` — `O(1)`. Vậy chỉ cần duyệt `n` học sinh → `O(n)`.

> Quy tắc: **Nếu** chỉ cần thông tin tại **một điểm**, **thì** kiểm tra trực tiếp từng đối tượng thay vì xây dựng toàn bộ trục.

Kiểm tra tay cho ví dụ 1:

In [9]:
q = 4
for i, (s, e) in enumerate(zip([1, 2, 3], [3, 2, 7])):
    print(f"học sinh {i}: {s} <= {q} <= {e} ? {s <= q <= e}")

học sinh 0: 1 <= 4 <= 3 ? False
học sinh 1: 2 <= 4 <= 2 ? False
học sinh 2: 3 <= 4 <= 7 ? True


## 5. Cách 2 (lời giải chính) — Duyệt và đếm

### 5.1. Mổ xẻ từng đoạn

```python
count = 0                                   # Đoạn 1: bộ đếm
for i in range(len(startTime)):             # Đoạn 2: duyệt từng học sinh
    if startTime[i] <= queryTime <= endTime[i]:
        count += 1                          # Đoạn 3: bận → đếm
return count
```

Viết theo chỉ số `i` trước để thấy rõ `startTime[i]` và `endTime[i]` **cùng thuộc một học sinh**; sau đó gọn hơn với `zip`.

In [10]:
def busy_loop(startTime, endTime, queryTime):
    count = 0
    for i in range(len(startTime)):
        if startTime[i] <= queryTime <= endTime[i]:
            count += 1
    return count

print(busy_loop([1, 2, 3], [3, 2, 7], 4))   # 1
print(busy_loop([4], [4], 4))               # 1

1
1


## 6. TRACE từng bước — ví dụ 1 (`queryTime = 4`)

| i | start | end | `start <= 4`? | `4 <= end`? | bận? | `count` sau |
|---:|---:|---:|---|---|---|---:|
| 0 | 1 | 3 | có | **không** | không | 0 |
| 1 | 2 | 2 | có | **không** | không | 0 |
| 2 | 3 | 7 | có | có | **có** | 1 |

In [11]:
def trace_busy(startTime, endTime, queryTime):
    count = 0
    for i, (s, e) in enumerate(zip(startTime, endTime)):
        left_ok = s <= queryTime
        right_ok = queryTime <= e
        busy = left_ok and right_ok
        count += busy
        print(f"i={i} [{s},{e}] start<=q:{left_ok!s:5} q<=end:{right_ok!s:5} bận:{busy!s:5} count={count}")
    return count

print("Kết quả:", trace_busy([1, 2, 3], [3, 2, 7], 4))

i=0 [1,3] start<=q:True  q<=end:False bận:False count=0
i=1 [2,2] start<=q:True  q<=end:False bận:False count=0
i=2 [3,7] start<=q:True  q<=end:True  bận:True  count=1
Kết quả: 1


`count += busy` cộng `True` (=1) hoặc `False` (=0) — chính là mẹo ở mục 2.3.

### 6.1. Trace các nhánh biên: `q` trùng đầu mút

In [12]:
print("Kết quả:", trace_busy([4, 1, 5], [9, 4, 6], 4))

i=0 [4,9] start<=q:True  q<=end:True  bận:True  count=1
i=1 [1,4] start<=q:True  q<=end:True  bận:True  count=2
i=2 [5,6] start<=q:False q<=end:True  bận:False count=2
Kết quả: 2


Học sinh 0 **bắt đầu** đúng lúc 4 và học sinh 1 **kết thúc** đúng lúc 4 — cả hai đều được tính. Học sinh 2 bắt đầu lúc 5 thì chưa.

## 7. Cách 3 — `zip` + `sum` một dòng

In [13]:
def busy_sum(startTime, endTime, queryTime):
    return sum(s <= queryTime <= e for s, e in zip(startTime, endTime))

print(busy_sum([1, 2, 3], [3, 2, 7], 4), busy_sum([4], [4], 4))

1 1


## 8. Cách 4 — Mảng hiệu (difference array) + prefix sum

Cải tiến cách 1: thay vì tô **cả đoạn**, chỉ đánh dấu **hai đầu**:
- `diff[start] += 1` — từ thời điểm `start` có thêm 1 người bận;
- `diff[end + 1] -= 1` — từ `end + 1` bớt 1 người.

Sau đó `busy[t] = diff[0] + ... + diff[t]` (prefix sum) = số người bận lúc `t`. Mỗi học sinh chỉ tốn `O(1)` thay vì `O(độ dài đoạn)`.

Cách này **thừa** nếu chỉ hỏi một `queryTime`, nhưng rất mạnh khi phải trả lời **nhiều truy vấn**: tiền xử lý `O(n + T)` một lần, sau đó mỗi truy vấn `O(1)`.

In [14]:
def build_busy_table(startTime, endTime, T=1000):
    diff = [0] * (T + 2)                 # cần chỗ cho end + 1 = T + 1
    for s, e in zip(startTime, endTime):
        diff[s] += 1
        diff[e + 1] -= 1
    return list(accumulate(diff))       # busy[t] với t = 0..T+1

def busy_diff(startTime, endTime, queryTime):
    return build_busy_table(startTime, endTime)[queryTime]

table = build_busy_table([1, 2, 3], [3, 2, 7], T=10)
print("t     :", list(range(12)))
print("busy  :", table)
print("lúc 4 :", table[4], "| hàm:", busy_diff([1, 2, 3], [3, 2, 7], 4))

t     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
busy  : [0, 1, 2, 2, 1, 1, 1, 1, 0, 0, 0, 0]
lúc 4 : 1 | hàm: 1


In [15]:
def trace_diff(startTime, endTime, T):
    diff = [0] * (T + 2)
    for s, e in zip(startTime, endTime):
        diff[s] += 1
        diff[e + 1] -= 1
        print(f"thêm [{s},{e}]: diff = {diff}")
    print("prefix  :", list(accumulate(diff)))

trace_diff([1, 2, 3], [3, 2, 7], T=8)

thêm [1,3]: diff = [0, 1, 0, 0, -1, 0, 0, 0, 0, 0]
thêm [2,2]: diff = [0, 1, 1, -1, -1, 0, 0, 0, 0, 0]
thêm [3,7]: diff = [0, 1, 1, 0, -1, 0, 0, 0, -1, 0]
prefix  : [0, 1, 2, 2, 1, 1, 1, 1, 0, 0]


## 9. Cách 5 — Sắp xếp + `bisect` (đếm bằng phép trừ)

Một học sinh bận lúc `q` ⇔ `start <= q` **và** `end >= q`. Học sinh **không** bận nhưng có `start <= q` chính là những người đã xong: `end < q` (vì `end >= start`, ai có `end < q` thì chắc chắn `start <= q`). Do đó:

$$\text{bận} = \#\{start \le q\} - \#\{end < q\}.$$

Sắp xếp hai mảng một lần, mỗi truy vấn chỉ cần 2 lần `bisect` → `O(log n)`.

In [16]:
def busy_bisect(startTime, endTime, queryTime):
    ss, es = sorted(startTime), sorted(endTime)
    started = bisect_right(ss, queryTime)      # start <= q
    finished = bisect_left(es, queryTime)      # end < q
    return started - finished

ss, es = sorted([1, 2, 3]), sorted([3, 2, 7])
print("start sorted:", ss, "| end sorted:", es)
print("đã bắt đầu (start<=4):", bisect_right(ss, 4), "| đã xong (end<4):", bisect_left(es, 4))
print("→ bận:", busy_bisect([1, 2, 3], [3, 2, 7], 4))

start sorted: [1, 2, 3] | end sorted: [2, 3, 7]
đã bắt đầu (start<=4): 3 | đã xong (end<4): 2
→ bận: 1


Chú ý: sắp xếp **riêng rẽ** hai mảng làm "mất" việc ghép cặp `(start, end)` theo học sinh — nhưng công thức chỉ cần **số lượng**, không cần biết ai là ai, nên vẫn đúng.

## 10. Code nộp LeetCode (Python 3)

In [17]:
class Solution:
    def busyStudent(self, startTime: list[int], endTime: list[int], queryTime: int) -> int:
        count = 0
        for s, e in zip(startTime, endTime):
            if s <= queryTime <= e:
                count += 1
        return count

sol = Solution()
print(sol.busyStudent([1, 2, 3], [3, 2, 7], 4))   # 1
print(sol.busyStudent([4], [4], 4))               # 1

1
1


Bản một dòng cũng Accepted:

```python
class Solution:
    def busyStudent(self, startTime: list[int], endTime: list[int], queryTime: int) -> int:
        return sum(s <= queryTime <= e for s, e in zip(startTime, endTime))
```

## 11. Vì sao thuật toán đúng?

- Theo đề, học sinh `i` được tính ⇔ `startTime[i] <= queryTime <= endTime[i]`. Thuật toán kiểm tra **đúng** điều kiện này cho **mỗi** `i` đúng một lần.
- **Invariant:** sau khi xét `i` học sinh đầu, `count` = số học sinh bận trong `i` người đó. Khởi tạo `0` đúng; mỗi bước cộng thêm 1 khi và chỉ khi người thứ `i` bận. Kết thúc: `count` = tổng số người bận.

Với cách 4: `busy[t] = Σ_{x<=t} diff[x]`. Học sinh `[s, e]` đóng góp `+1` khi `t >= s` và `-1` khi `t >= e+1`, nên đóng góp ròng là `1` đúng khi `s <= t <= e`, ngược lại `0`. Cộng mọi học sinh → số người bận lúc `t`.

Với cách 5: tập {start ≤ q} = {bận} ∪ {end < q}, hai tập con rời nhau (bận thì `end >= q`), và mọi người có `end < q` đều có `start <= end < q`. Nên `#bận = #{start ≤ q} − #{end < q}`.

## 12. Phân tích Time / Space

Gọi `n` = số học sinh, `T` = thời điểm lớn nhất (≤ 1000), `L` = độ dài đoạn trung bình.

| Cách | Một truy vấn | `Q` truy vấn | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng tô cả đoạn | `O(n·L + T)` | tiền xử lý `O(n·L + T)`, mỗi truy vấn `O(1)` | `O(T)` | chậm |
| 2. Duyệt chỉ số | `O(n)` | `O(Q·n)` | `O(1)` | **khuyên dùng cho bài này** |
| 3. `zip` + `sum` | `O(n)` | `O(Q·n)` | `O(1)` | gọn |
| 4. Mảng hiệu | `O(n + T)` | `O(n + T + Q)` | `O(T)` | tốt khi nhiều truy vấn, `T` nhỏ |
| 5. Sort + bisect | `O(n log n)` | `O(n log n + Q log n)` | `O(n)` | tốt khi nhiều truy vấn, `T` lớn |

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.busyStudent([1, 2, 3], [3, 2, 7], 4), 1)
check("ví dụ 2", sol.busyStudent([4], [4], 4), 1)
check("q trùng start", sol.busyStudent([4], [9], 4), 1)
check("q trùng end", sol.busyStudent([1], [4], 4), 1)
check("q ngay trước start", sol.busyStudent([5], [9], 4), 0)
check("q ngay sau end", sol.busyStudent([1], [3], 4), 0)
check("mọi người bận", sol.busyStudent([1, 1, 1], [1000, 1000, 1000], 500), 3)
check("không ai bận", sol.busyStudent([1, 2], [2, 3], 1000), 0)
check("biên giá trị 1 và 1000", sol.busyStudent([1, 1000], [1, 1000], 1000), 1)

PASS  ví dụ 1: 1
PASS  ví dụ 2: 1
PASS  q trùng start: 1
PASS  q trùng end: 1
PASS  q ngay trước start: 0
PASS  q ngay sau end: 0
PASS  mọi người bận: 3
PASS  không ai bận: 0
PASS  biên giá trị 1 và 1000: 1


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

In [19]:
import random
rng = random.Random(1450)
methods = [busy_timeline, busy_loop, busy_sum, busy_diff, busy_bisect, Solution().busyStudent]
for _ in range(500):
    n = rng.randint(1, 100)
    st, en = [], []
    for _ in range(n):
        s = rng.randint(1, 1000)
        e = rng.randint(s, min(1000, s + rng.choice([0, 5, 50, 1000])))
        st.append(s); en.append(e)
    q = rng.choice([rng.randint(1, 1000), rng.choice(st), rng.choice(en)])   # hay chọn đúng đầu mút
    results = {f(st, en, q) for f in methods}
    assert len(results) == 1, (st, en, q, results)
print("500 bộ ngẫu nhiên: 6 hàm cho cùng kết quả")

500 bộ ngẫu nhiên: 6 hàm cho cùng kết quả


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Dùng dấu `<` (bỏ mất đầu mút)

In [20]:
def wrong_strict(startTime, endTime, q):
    return sum(s < q < e for s, e in zip(startTime, endTime))

print("Sai:", wrong_strict([4], [4], 4), "| Đúng:", busy_sum([4], [4], 4))

Sai: 0 | Đúng: 1


**Vì sao sai:** đề nói "inclusive". **Sửa:** `s <= q <= e`.

### Lỗi 2 — Dùng `or` thay vì `and`

In [21]:
def wrong_or(startTime, endTime, q):
    return sum((s <= q) or (q <= e) for s, e in zip(startTime, endTime))

print("Sai:", wrong_or([1, 2, 3], [3, 2, 7], 4), "| Đúng:", busy_sum([1, 2, 3], [3, 2, 7], 4))

Sai: 3 | Đúng: 1


**Vì sao sai:** "đã bắt đầu **hoặc** chưa xong" đúng với gần như mọi người. Nằm trong đoạn cần **cả hai** điều kiện.

### Lỗi 3 — Viết sai chained comparison: `s <= q >= e`

`s <= q >= e` nghĩa là `s <= q and q >= e` — tức "đã xong", không phải "đang làm".

In [22]:
s, q, e = 3, 4, 7
print("s <= q >= e :", s <= q >= e, "(sai ý)")
print("s <= q <= e :", s <= q <= e, "(đúng ý)")

s <= q >= e : False (sai ý)
s <= q <= e : True (đúng ý)


### Lỗi 4 — Mảng hiệu thiếu một ô

`diff = [0] * (T + 1)` rồi `diff[e + 1] -= 1` với `e = 1000` → `IndexError`. **Sửa:** cấp `T + 2` ô.

In [23]:
try:
    diff = [0] * (1000 + 1)
    diff[1000 + 1] -= 1
except IndexError as err:
    print("Lỗi:", err)

Lỗi: list index out of range


### Lỗi 5 — `range(s, e)` khi mô phỏng (quên `+ 1`)

`range(s, e)` không gồm `e`, nên thời điểm kết thúc bị bỏ sót:

In [24]:
def wrong_range(startTime, endTime, q):
    timeline = [0] * 1001
    for s, e in zip(startTime, endTime):
        for t in range(s, e):
            timeline[t] += 1
    return timeline[q]

print("Sai:", wrong_range([1], [4], 4), "| Đúng:", busy_timeline([1], [4], 4))

Sai: 0 | Đúng: 1


## 15. Mẹo nhận dạng pattern

- Dữ liệu dạng **hai mảng song song** (`start[i]`, `end[i]`) → `zip`.
- "Điểm `q` thuộc bao nhiêu đoạn?" với **một** truy vấn → duyệt `O(n)`.
- **Nhiều** truy vấn hoặc hỏi "thời điểm đông nhất" → **mảng hiệu + prefix sum** (khi trục thời gian nhỏ) hoặc **sweep line / sort + bisect** (khi trục lớn).
- Cẩn thận **đoạn đóng hay nửa mở**: `[s, e]` hay `[s, e)` — đổi `<=` thành `<` ở đầu nào.
- Bài cùng họ: *Car Pooling* (1094), *Corporate Flight Bookings* (1109), *Meeting Rooms II* (253), *Range Addition* (370).

## 16. Biến thể

### 16.1. Nhiều truy vấn cùng lúc

In [25]:
def busy_many_queries(startTime, endTime, queries):
    table = build_busy_table(startTime, endTime)
    return [table[q] for q in queries]

print(busy_many_queries([1, 2, 3], [3, 2, 7], [1, 2, 3, 4, 7, 8]))

[1, 2, 2, 1, 1, 0]


### 16.2. Thời điểm đông nhất và số người lúc đó

In [26]:
def busiest_time(startTime, endTime):
    table = build_busy_table(startTime, endTime)
    best_t = max(range(len(table)), key=lambda t: table[t])   # t nhỏ nhất đạt max
    return best_t, table[best_t]

print(busiest_time([1, 2, 3], [3, 2, 7]))   # (2, 2): lúc 2 có học sinh 0 và 1

(2, 2)


`max(range(...), key=lambda t: table[t])` chọn `t` có `table[t]` lớn nhất; `lambda t: table[t]` là hàm vô danh nhận `t` trả `table[t]`. Khi hòa, `max` giữ phần tử **đầu tiên** gặp.

### 16.3. Đoạn nửa mở `[start, end)` — học sinh nộp bài lúc `end` thì không tính

In [27]:
def busy_half_open(startTime, endTime, q):
    return sum(s <= q < e for s, e in zip(startTime, endTime))

print(busy_half_open([1, 2, 3], [3, 2, 7], 3), "vs đóng:", busy_sum([1, 2, 3], [3, 2, 7], 3))

1 vs đóng: 2


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** `startTime = [2, 5, 5, 9]`, `endTime = [6, 5, 10, 9]`. Đếm số học sinh bận lúc `q = 5` và `q = 9`.

**Bài 2 — Dự đoán output.**
```python
print(1 <= 5 <= 3, 1 < 2 < 3 < 4, 5 >= 3 <= 4)
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải đếm

In [28]:
def exercise_busy(startTime, endTime, queryTime):
    # TODO: zip + kiểm tra đoạn đóng + đếm
    pass

# assert exercise_busy([1,2,3], [3,2,7], 4) == 1
# assert exercise_busy([4], [4], 4) == 1

### Bài 4 — TODO: danh sách chỉ số học sinh đang bận

In [29]:
def exercise_busy_ids(startTime, endTime, queryTime):
    # TODO: trả list các i (tăng dần) thỏa start <= q <= end
    pass

# assert exercise_busy_ids([4, 1, 5], [9, 4, 6], 4) == [0, 1]

### Bài 5 — TODO: Car Pooling (LeetCode 1094)
`trips[i] = [numPassengers, from, to]`: xe đón khách ở km `from`, trả khách ở km `to` (lúc `to` khách đã xuống — **nửa mở** `[from, to)`). Xe chở tối đa `capacity`. Trả `True` nếu chở được mọi chuyến. `0 <= from < to <= 1000`.

In [30]:
def exercise_car_pooling(trips, capacity):
    # TODO: mảng hiệu: diff[from] += num, diff[to] -= num; prefix không vượt capacity
    pass

# assert exercise_car_pooling([[2,1,5],[3,3,7]], 4) == False
# assert exercise_car_pooling([[2,1,5],[3,3,7]], 5) == True
# assert exercise_car_pooling([[2,1,5],[3,5,7]], 3) == True

### Bài 6 — Mini challenge: Q truy vấn với thời gian tới `10^9`
Thời gian có thể tới `10^9` nên không tạo được mảng hiệu. Trả list đáp án cho mọi truy vấn, nhanh hơn `O(Q·n)`.

In [31]:
def exercise_busy_big(startTime, endTime, queries):
    # TODO: sort hai mảng một lần, mỗi truy vấn dùng bisect
    pass

# assert exercise_busy_big([1, 10**9], [5, 10**9], [1, 5, 6, 10**9]) == [1, 1, 0, 1]

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Mỗi học sinh: một phép kiểm tra. | `zip` + biến đếm hoặc `sum`. | `sum(s <= q <= e for s, e in zip(...))` |
| 4 | Cần chỉ số → `enumerate`. | list comprehension có `if`. | `[i for i, (s, e) in enumerate(zip(...)) if s <= q <= e]` |
| 5 | Số khách trên xe tại mỗi km = prefix sum. | `diff` dài 1001 (hoặc 1002). | duyệt `accumulate(diff)`, nếu `> capacity` trả `False` |
| 6 | `#bận = #{start<=q} − #{end<q}`. | `sorted` một lần, `bisect_right`/`bisect_left`. | `bisect_right(ss, q) - bisect_left(es, q)` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `q = 5`: các đoạn `[2,6]`, `[5,5]`, `[5,10]` → **3**; `q = 9`: `[5,10]`, `[9,9]` → **2**.
**Bài 2:** `False True True` (`5 >= 3 <= 4` là `5 >= 3 and 3 <= 4`).

In [32]:
def answer_busy(startTime, endTime, queryTime):
    return sum(s <= queryTime <= e for s, e in zip(startTime, endTime))

def answer_busy_ids(startTime, endTime, queryTime):
    return [i for i, (s, e) in enumerate(zip(startTime, endTime)) if s <= queryTime <= e]

def answer_car_pooling(trips, capacity):
    diff = [0] * 1002
    for num, frm, to in trips:
        diff[frm] += num
        diff[to] -= num
    return all(load <= capacity for load in accumulate(diff))

def answer_busy_big(startTime, endTime, queries):
    ss, es = sorted(startTime), sorted(endTime)
    return [bisect_right(ss, q) - bisect_left(es, q) for q in queries]

assert answer_busy([1,2,3], [3,2,7], 4) == 1
assert answer_busy([4], [4], 4) == 1
assert answer_busy_ids([4, 1, 5], [9, 4, 6], 4) == [0, 1]
assert answer_car_pooling([[2,1,5],[3,3,7]], 4) == False
assert answer_car_pooling([[2,1,5],[3,3,7]], 5) == True
assert answer_car_pooling([[2,1,5],[3,5,7]], 3) == True
assert answer_busy_big([1, 10**9], [5, 10**9], [1, 5, 6, 10**9]) == [1, 1, 0, 1]
print("Bài 1:", answer_busy([2,5,5,9], [6,5,10,9], 5), answer_busy([2,5,5,9], [6,5,10,9], 9))
print("Bài 2:", 1 <= 5 <= 3, 1 < 2 < 3 < 4, 5 >= 3 <= 4)
print("Đáp án bài 3–6: PASS")

Bài 1: 3 2
Bài 2: False True True
Đáp án bài 3–6: PASS


## 20. Template nhớ nhanh + checklist submit

```python
# Một truy vấn: điểm q thuộc bao nhiêu đoạn đóng [s, e]?
count = sum(s <= q <= e for s, e in zip(starts, ends))

# Nhiều truy vấn, trục nhỏ: mảng hiệu
diff[s] += 1; diff[e + 1] -= 1   →   busy = list(accumulate(diff))

# Nhiều truy vấn, trục lớn: sort + bisect
bisect_right(sorted_starts, q) - bisect_left(sorted_ends, q)
```

**Checklist trước khi nộp**
- [ ] Đoạn **đóng**: dùng `<=` ở cả hai phía.
- [ ] Điều kiện là **và** (`and` / chained comparison), không phải `or`.
- [ ] Ghép đúng `startTime[i]` với `endTime[i]` (dùng `zip`).
- [ ] Nếu dùng mảng hiệu: đủ `T + 2` ô.
- [ ] Method tên `busyStudent`, trả về `int`.